functools is a built-in Python module that provides higher-order functions — tools that either:

* take functions as input,
* return functions,
* or modify function behavior.

 These basically are collection of tools that simplify working with functions and callable objects.

#**Partial**

The partial class fix certain arguments of a function and create a new function with fewer parameters. This is especially useful for creating specialized versions of functions without defining new ones from scratch.

## Syntax:
partial (afunction,*args, **kwargs)

*args are Positional arguments to fix from left to right.

In [ ]:
from functools import partial
def power(a, b):
    return a ** b

square = partial(power, b=2) # specialized version of power function for square
pow4 = partial(power, b=4)
power_of_5 = partial(power, 5)

print(power(2, 3)) # raw implementation of power fn
print(square(4))
print(pow4(3))
print(power_of_5(2))


#partial.sth..
print(square.func) # name of fn along with address
print(square.keywords) #shows the keyword argument passed to the function
print(power_of_5.args) #shows the positional argument  or simply arguments passed to the function

8
16
81
25
<function power at 0x7dd91e766340>
{'b': 2}
(5,)


# **partialmethod**

it is like partial but for classes

In [ ]:
from functools import partialmethod

class Demo:
    def __init__(self):
        self.color = 'black'

    def _color(self, type):
        self.color = type

    set_red = partialmethod(_color, type='red')
    set_blue = partialmethod(_color, type='blue')
    set_green = partialmethod(_color, type='green')

obj = Demo()
print(obj.color)
obj.set_blue()
print(obj.color)

black
blue


but i can do the same using partial

In [ ]:
from functools import partial
class Demo_using_partial:
    def __init__(self):
        self.color = 'black'

    def _color(self, type):
        self.color = type

    set_red = partial(_color, type='red')
    set_blue = partial(_color, type='blue')
    set_green = partial(_color, type='green')

obj = Demo_using_partial()
print(obj.color)
# obj.set_blue() # required positional argument: 'self' vanxa so i circumvented as:
Demo_using_partial.set_blue(obj)
obj.set_blue(obj)
print(obj.color)
print(obj.color)
print()

black
blue
blue



# **lru_cache**

## **Syntax**
@lru_cache(maxsize=128, typed=False)

* maxsize: Maximum number of recent calls to cache (default is 128). Use None for unlimited cache.
* typed: If True, arguments of different types are cached separately (e.g., 1 and 1.0).

In [ ]:
from functools import lru_cache

@lru_cache(maxsize=None)
def factorial(n):
    if n <= 1:
        return 1
    return n * factorial(n-1)

print([factorial(n) for n in range(7)])
print(factorial.cache_info())

@lru_cache(maxsize=None)
def fib(n):
    if n < 2:
        return n
    return fib(n-1) + fib(n-2)

print([fib(n) for n in range(16)])
print(fib.cache_info())

[1, 1, 2, 6, 24, 120, 720]
CacheInfo(hits=5, misses=7, maxsize=None, currsize=7)
[0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144, 233, 377, 610]
CacheInfo(hits=28, misses=16, maxsize=None, currsize=16)


In [ ]:
from functools import lru_cache
import time

@lru_cache(maxsize=None)
def mult_by_3(n):
    time.sleep(2)  # sleep for 2 seconds
    return n * 3

# First round - all will take ~2 seconds each
print("First round:")
start = time.time()
print(mult_by_3(1))
print(f"done for 1 - took {time.time() - start} seconds")
print(f"done for 1 - took {time.time() - start:.2f} seconds\n")

start = time.time()
print(mult_by_3(2))
print(f"done for 2 - took {time.time() - start:.2f} seconds\n")

start = time.time()
print(mult_by_3(3))
print(f"done for 3 - took {time.time() - start:.2f} seconds\n")

# Second round - these will be instant (cached)
print("--Again--")
start = time.time()
print(mult_by_3(1))
print(f"done for 1 - took {time.time() - start} seconds")
print(f"done for 1 - took {time.time() - start:.4f} seconds\n")

start = time.time()
print(mult_by_3(2))
print(f"done for 2 - took {time.time() - start:.4f} seconds\n")

start = time.time()
print(mult_by_3(3))
print(f"done for 3 - took {time.time() - start:.4f} seconds\n")

# New value - will take ~2 seconds
print("\n--New Value--")
start = time.time()
print(mult_by_3(33))
print(f"done for 33 - took {time.time() - start:.2f} seconds\n")

First round:
3
done for 1 - took 2.0006520748138428 seconds
done for 1 - took 2.00 seconds

6
done for 2 - took 2.00 seconds

9
done for 3 - took 2.00 seconds

--Again--
3
done for 1 - took 0.00010538101196289062 seconds
done for 1 - took 0.0002 seconds

6
done for 2 - took 0.0001 seconds

9
done for 3 - took 0.0001 seconds


--New Value--
99
done for 33 - took 2.00 seconds



# wraps
wraps is a decorator that applies update_wrapper automatically. It’s commonly used when writing decorators to preserve original function metadata.

## **Why is it used?**

According to documentation, The main intended use for this function is in decorator functions which wrap the decorated function and return the wrapper. If the wrapper function is not updated, the metadata of the returned function will reflect the wrapper definition rather than the original function definition, which is typically less than helpful.

### Problem

In [ ]:
def my_decorator(func):
    '''The wrapping function'''
    def wrapper():
        '''Not the wanted docstring'''
        print("Before")
        func()
    print("Wrapper Docstring:",wrapper.__doc__)
    return wrapper

@my_decorator
def greet():
    '''This doc string is lost'''
    pass

print(greet.__name__)
print(greet.__doc__)

Wrapper Docstring: Not the wanted docstring
wrapper
Not the wanted docstring


### Solution

In [ ]:
from functools import wraps

def my_decorator(func):
    '''The wrapping function'''
    @wraps(func)
    def wrapper():
        '''Not the wanted docstring'''
        print("Before")
        func()
    return wrapper

@my_decorator
# if my_decorator() took some arguments say a and bthen the decorator would be:
# @my_decorator(a = 5,b=3) #or @my_decorator(5,3)
def greet():
    '''This doc string is shown'''
    pass

print(greet.__name__)
print(greet.__doc__)

greet
This doc string is shown


# **update_wrapper**
update_wrapper updates a wrapper function to copy attributes (\_\_name__, \_\_doc__, etc.) from the wrapped function.

In [ ]:
from functools import update_wrapper, partial

def power(a, b):
    '''a to the power b'''
    return a ** b

pow2 = partial(power, b=2)
pow2.__doc__ = 'a to the power 2'
pow2.__name__ = 'pow2'

print('Before update:')
print('Doc:', pow2.__doc__)
print('Name:', pow2.__name__)

update_wrapper(pow2, power)

print('\nAfter update:')
print('Doc:', pow2.__doc__)
print('Name:', pow2.__name__)

Before update:
Doc: a to the power 2
Name: pow2

After update:
Doc: a to the power b
Name: power


# cmp_to_key

it is a old school method

Cmp_to_key converts a comparison function into a key function. The comparison function must return 1, -1 and 0 for different conditions.

Used with tools that accept key functions (such as sorted(), min(), max(), heapq.nlargest(), heapq.nsmallest(), itertools.groupby())

In [ ]:
from functools import cmp_to_key

def cmp_fun(a, b):
    if a[-1] > b[-1]:
        return 1
    elif a[-1] < b[-1]:
        return -1
    else:
        return 0

def sort_by_length_des(a,b):
  if len(a)>len(b):
      return -1

  elif len(a)<len(b):
      return 1

  else:
      return 0


l1 = ['ali', 'ranveer', 'dipika']
sorted_list = sorted(l1, key=cmp_to_key(cmp_fun))
print('Sorted list by last letter:', sorted_list)
sorted_list = sorted(l1, key=cmp_to_key(sort_by_length_des))
print('Sorted list by length:', sorted_list)

Sorted list by last letter: ['dipika', 'ali', 'ranveer']
Sorted list by length: ['ranveer', 'dipika', 'ali']


# **Reduce**

combines the iterables into a single value

## Syntax

reduce(function, iterable[, initializer])

* initializer (optional): A starting value placed before the items of iterable

In [ ]:
from functools import reduce
l1 = [2, 4, 7, 9, 1, 3]
l2 = ["abc", "xyz", "def"]

sum_of_list1 = reduce(lambda a, b:a + b, l1)
max_of_list2 = reduce(lambda a, b:a if a>b else b, l2)

print('Sum of list1 :', sum_of_list1)
print('Maximum of list2 :', max_of_list2)

Sum of list1 : 26
Maximum of list2 : xyz


# total_ordering

This class decorator automatically fills in missing comparison methods (\_\_lt__, \_\_gt__, etc.) based on the few you provide.

In [ ]:
from functools import total_ordering

@total_ordering
class N:
    def __init__(self, value):
        self.value = value

    def __eq__(self, other):
        return self.value == other.value

    def __lt__(self, other):
        return self.value < other.value

#notice we have only given two operators to check but python fills missing comparision methods

print('6 > 2:', N(6) > N(2))
print('3 < 1:', N(3) < N(1))
print('2 <= 7:', N(2) <= N(7))
print('9 >= 10:', N(9) >= N(10))
print('5 == 5:', N(5) == N(5))

6 > 2: True
3 < 1: False
2 <= 7: True
9 >= 10: False
5 == 5: True


# **singledispatch**

 it turns a function into a generic function that dispatches calls to different implementations based on the type of the first argument.

## Why is it used?
python doesnt support Type-based Function Overloading so it helps to simulate it.

instead of doing:

  if isinstance(x, int):

      call_integerfn(x)

we can directly use sinle dispacthed function

In [ ]:
from functools import singledispatch

@singledispatch
def process(data):
    print(f"Unspecified type: {data}")

@process.register(int)
def _(a):
    print(f"Integer type {a}")

@process.register(float)
def _(data):
    print(f"Float type {data}")

@process.register(str)
def _(b):
    print(f"String type {b}")

process(10)
process("hello")
process(10.1)
process([10.1,"Hello", "Namaste", 5, False])

Integer type 10
String type hello
Float type 10.1
Unspecified type: [10.1, 'Hello', 'Namaste', 5, False]
